# NAIP ZIP → SDR-Ready Processing Pipeline

This notebook takes a folder of downloaded USGS NAIP `.ZIP` files and prepares them for accession into the **Stanford Digital Repository (SDR)**.

### Pipeline Overview

| Step | Action |
|------|--------|
| 1 | Install dependencies & import libraries |
| 2 | Configure directory paths and COG creation settings |
| 3 | Discover and inventory all ZIP files |
| 4 | Extract ZIPs into per-scene subfolders (originals retained) |
| 5 | Catalog extracted folder structures and file types |
| 6 | Parse ISO 19115 XML metadata from each scene |
| 7 | Validate each raster — is it already a Cloud-Optimized GeoTIFF (COG)? |
| 8 | Convert non-COG images to COGs |
| 9 | Replace originals with COGs in extracted folders only |
| 10 | Stage original (non-COG) images for QAQC disposal |
| 11 | Build a STAC Item for each dataset |
| 12 | Create and validate a top-level STAC Catalog |
| 13 | Final verification report |

**Key invariant:** Original `.ZIP` files are **never modified**. Only the extracted copies receive COG replacements.

## Step 1 — Install Dependencies & Import Libraries

| Package | Role |
|---------|------|
| `rasterio` | Raster I/O, CRS extraction, band metadata |
| `rio-cogeo` | COG validation (`cog_validate`) and creation (`cog_translate`) |
| `pystac` | Build STAC Catalog / Items conforming to the STAC spec |
| `shapely` | Bounding-box → GeoJSON geometry conversion |
| `tqdm` | Progress bars |
| `lxml` | Robust ISO 19115 XML parsing |

Standard-library modules used: `zipfile`, `pathlib`, `shutil`, `json`, `hashlib`, `datetime`, `csv`, `xml.etree.ElementTree`.

In [ ]:
# ── Install Dependencies ──────────────────────────────────────────────────────
!pip install rasterio rio-cogeo pystac shapely tqdm lxml --quiet

In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import os
import json
import csv
import hashlib
import shutil
import zipfile
from pathlib import Path
from datetime import datetime, timezone
from xml.etree import ElementTree as ET

import rasterio
from rasterio.warp import transform_bounds
from rio_cogeo.cogeo import cog_translate, cog_validate
from rio_cogeo.profiles import cog_profiles
import pystac
from pystac import Catalog, Item, Asset, MediaType
from shapely.geometry import box, mapping
from tqdm.auto import tqdm

print("✔ All imports loaded.")

## Step 2 — Define Directory Paths & Configuration

| Variable | Purpose |
|----------|---------|
| `DOWNLOAD_DIR` | Root folder containing the downloaded `.ZIP` files |
| `ORIGINALS_DIR` | Parallel folder for staging original (non-COG) images before disposal |
| `STAC_CATALOG_ID` | Identifier string for the STAC catalog |
| `COG_PROFILE` | `rio-cogeo` compression profile (`lzw`, `deflate`, `zstd`, etc.) |
| `COG_BLOCKSIZE` | Tile size for COG internal tiling (default 512) |
| `OVERVIEW_RESAMPLING` | Resampling method for overview pyramids |
| `IMAGE_EXTENSIONS` | File suffixes treated as raster imagery |

In [ ]:
# ── Directory Paths ───────────────────────────────────────────────────────────
# Point this at the folder that contains the downloaded .ZIP files.
DOWNLOAD_DIR = Path("./downloads/naip")

# Original (non-COG) images are staged here for QAQC review before deletion.
# This folder sits *parallel* to the downloads folder.
ORIGINALS_DIR = DOWNLOAD_DIR.parent / "originals_for_disposal"
ORIGINALS_DIR.mkdir(parents=True, exist_ok=True)

# ── Image file extensions to treat as raster imagery ─────────────────────────
IMAGE_EXTENSIONS = {".tif", ".tiff", ".img", ".jp2", ".ntf"}

# ── STAC Catalog settings ────────────────────────────────────────────────────
STAC_CATALOG_ID   = "naip-sdr-catalog"
STAC_CATALOG_TITLE = "NAIP Imagery — SDR Accession"
STAC_CATALOG_DESC  = (
    "Cloud-Optimized GeoTIFF imagery derived from USGS NAIP downloads, "
    "prepared for ingest into the Stanford Digital Repository."
)

# ── COG creation profile ─────────────────────────────────────────────────────
COG_PROFILE_NAME    = "lzw"        # rio-cogeo profile: lzw | deflate | zstd | raw
COG_BLOCKSIZE       = 512          # internal tile size (pixels)
OVERVIEW_RESAMPLING = "nearest"    # nearest | bilinear | cubic | average
OVERVIEW_LEVELS     = [2, 4, 8, 16, 32]  # overview decimation levels

print(f"Download dir    : {DOWNLOAD_DIR.resolve()}")
print(f"Originals dir   : {ORIGINALS_DIR.resolve()}")
print(f"COG profile     : {COG_PROFILE_NAME}, block={COG_BLOCKSIZE}, "
      f"overviews={OVERVIEW_LEVELS}")
print(f"Image extensions: {IMAGE_EXTENSIONS}")

## Step 3 — Discover & Inventory ZIP Files

Scan the download directory for all `.zip` / `.ZIP` files and build an inventory with file size and modification time. This inventory is the master list used for the rest of the pipeline.

In [ ]:
# ── Discover ZIP files ────────────────────────────────────────────────────────

def sha256_file(path: Path, chunk_size: int = 1 << 20) -> str:
    """Return the SHA-256 hex digest of a file (read in chunks)."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(chunk_size):
            h.update(chunk)
    return h.hexdigest()


zip_files = sorted(
    p for p in DOWNLOAD_DIR.iterdir()
    if p.suffix.lower() == ".zip" and p.is_file()
)

# Build inventory with pre-processing checksums (used later to verify ZIPs are untouched)
zip_inventory: list[dict] = []
print(f"Computing SHA-256 checksums for {len(zip_files)} ZIP file(s) …")
for zf in tqdm(zip_files, desc="Checksumming ZIPs"):
    stat = zf.stat()
    zip_inventory.append({
        "path": zf,
        "name": zf.name,
        "size_mb": stat.st_size / (1024 * 1024),
        "modified": datetime.fromtimestamp(stat.st_mtime).isoformat(),
        "sha256": sha256_file(zf),
    })

total_mb = sum(z["size_mb"] for z in zip_inventory)
print(f"\n{'═' * 60}")
print(f" ZIP Inventory")
print(f"{'═' * 60}")
print(f"  Files : {len(zip_inventory)}")
print(f"  Total : {total_mb:,.1f} MB  ({total_mb / 1024:.2f} GB)")
print(f"{'═' * 60}")

# Show first few entries
for z in zip_inventory[:5]:
    print(f"  {z['name']:<55} {z['size_mb']:>8.1f} MB")
if len(zip_inventory) > 5:
    print(f"  … and {len(zip_inventory) - 5} more")

## Step 4 — Extract ZIP Files (Retain Originals)

Each ZIP is extracted into a subfolder named after the ZIP file (minus the `.ZIP` extension). The original ZIP files are **never modified**.

For example:
```
downloads/naip/
├── m_3611842_sw_11_060_20220704.ZIP          ← untouched
├── m_3611842_sw_11_060_20220704/             ← extracted contents
│   ├── m_3611842_sw_11_060_20220704.tif
│   └── m_3611842_sw_11_060_20220704.xml
└── …
```

In [ ]:
# ── Extract ZIPs ──────────────────────────────────────────────────────────────

extraction_results: dict[str, dict] = {}   # name → {"dest": Path, "files": [], "error": str|None}

for z in tqdm(zip_inventory, desc="Extracting ZIPs"):
    zpath: Path = z["path"]
    dest = DOWNLOAD_DIR / zpath.stem      # folder named after the ZIP (minus extension)
    result = {"dest": dest, "files": [], "error": None}

    try:
        with zipfile.ZipFile(zpath, "r") as zf:
            # Check for bad ZIP
            bad = zf.testzip()
            if bad:
                result["error"] = f"Corrupt member: {bad}"

            zf.extractall(dest)
            result["files"] = zf.namelist()

    except Exception as exc:
        result["error"] = str(exc)

    extraction_results[z["name"]] = result

# ── Summary ──────────────────────────────────────────────────────────────────
ok    = [k for k, v in extraction_results.items() if v["error"] is None]
bad   = [k for k, v in extraction_results.items() if v["error"] is not None]
total_files = sum(len(v["files"]) for v in extraction_results.values())

print(f"\n{'═' * 60}")
print(f" Extraction Summary")
print(f"{'═' * 60}")
print(f"  ZIP files processed : {len(extraction_results)}")
print(f"  Successful          : {len(ok)}")
print(f"  Failed / warnings   : {len(bad)}")
print(f"  Total files extracted: {total_files}")
print(f"{'═' * 60}")

if bad:
    print("\n⚠ Problems:")
    for name in bad:
        print(f"  {name}: {extraction_results[name]['error']}")

## Step 5 — Catalog Extracted Folder Structures

Walk each extracted folder and classify every file by type: **image**, **metadata** (XML/JSON), **sidecar** (PRJ, DBF, SHX, etc.), or **other**. This gives a complete inventory of what each scene contains before we start modifying anything.

In [ ]:
# ── Catalog extracted file structures ─────────────────────────────────────────

METADATA_EXTENSIONS = {".xml", ".json", ".csv", ".txt", ".htm", ".html"}
SIDECAR_EXTENSIONS  = {".prj", ".dbf", ".shx", ".shp", ".cpg", ".sbn", ".sbx",
                       ".ovr", ".aux", ".aux.xml", ".vat.dbf"}

def classify_file(path: Path) -> str:
    """Return a category string for a file based on its extension."""
    suffix = path.suffix.lower()
    # Check multi-part extensions first (.aux.xml, .vat.dbf)
    double_suffix = "".join(path.suffixes[-2:]).lower() if len(path.suffixes) >= 2 else ""
    if double_suffix in SIDECAR_EXTENSIONS:
        return "sidecar"
    if suffix in IMAGE_EXTENSIONS:
        return "image"
    if suffix in METADATA_EXTENSIONS:
        return "metadata"
    if suffix in SIDECAR_EXTENSIONS:
        return "sidecar"
    return "other"


# Build a per-dataset file catalog
dataset_catalog: dict[str, dict] = {}  # dataset_name → {"folder": Path, "files": {category: [paths]}}

for name, info in extraction_results.items():
    if info["error"] is not None:
        continue
    dest: Path = info["dest"]
    if not dest.exists():
        continue

    catalog_entry = {"folder": dest, "files": {"image": [], "metadata": [], "sidecar": [], "other": []}}

    for root, _dirs, files in os.walk(dest):
        for fname in files:
            fpath = Path(root) / fname
            cat = classify_file(fpath)
            catalog_entry["files"][cat].append(fpath)

    dataset_name = dest.name
    dataset_catalog[dataset_name] = catalog_entry

# ── Summary ──────────────────────────────────────────────────────────────────
print(f"{'═' * 70}")
print(f" Extracted Dataset Catalog  ({len(dataset_catalog)} datasets)")
print(f"{'═' * 70}")
print(f"  {'Dataset':<45} {'Images':<8} {'Meta':<6} {'Side':<6} {'Other':<6}")
print(f"  {'─' * 45} {'─' * 8} {'─' * 6} {'─' * 6} {'─' * 6}")

total_images = 0
for ds_name, entry in sorted(dataset_catalog.items()):
    fi = entry["files"]
    ni = len(fi["image"])
    total_images += ni
    print(f"  {ds_name:<45} {ni:<8} {len(fi['metadata']):<6} "
          f"{len(fi['sidecar']):<6} {len(fi['other']):<6}")

print(f"{'═' * 70}")
print(f"  Total image files: {total_images}")

## Step 6 — Parse ISO 19115 XML Metadata

Each NAIP ZIP contains an ISO 19115 XML sidecar. The parser extracts:

| Field | XML Path |
|-------|----------|
| Title | `gmd:identificationInfo / … / gmd:title` |
| Abstract | `gmd:identificationInfo / … / gmd:abstract` |
| Bounding box (W/E/S/N) | `gmd:EX_GeographicBoundingBox` |
| CRS (EPSG code) | `gmd:referenceSystemInfo / … / gmd:code` |
| Publication date | `gmd:dateStamp` |
| Resolution | `gmd:axisDimensionProperties / … / gmd:resolution` |

Where XML metadata is missing, rasterio reads CRS, bounds, resolution, and band count directly from the raster file.

In [ ]:
# ── Parse ISO 19115 XML metadata ──────────────────────────────────────────────

# ISO 19115 XML namespaces
NS = {
    "gmd": "http://www.isotc211.org/2005/gmd",
    "gco": "http://www.isotc211.org/2005/gco",
    "gmi": "http://www.isotc211.org/2005/gmi",
    "gml": "http://www.opengis.net/gml/3.2",
}


def _text(el, xpath: str) -> str | None:
    """Find first matching element and return its text, or None."""
    node = el.find(xpath, NS)
    if node is not None and node.text:
        return node.text.strip()
    return None


def parse_iso_xml(xml_path: Path) -> dict:
    """Parse an ISO 19115 XML file and return a flat metadata dict."""
    tree = ET.parse(xml_path)
    root = tree.getroot()

    meta: dict = {"source_xml": str(xml_path)}

    # Title
    meta["title"] = _text(root, ".//gmd:identificationInfo//gmd:citation//gmd:title/gco:CharacterString")

    # Abstract
    meta["abstract"] = _text(root, ".//gmd:identificationInfo//gmd:abstract/gco:CharacterString")

    # Publication date
    meta["date_published"] = _text(root, ".//gmd:dateStamp/gco:Date")

    # CRS (EPSG code string, e.g. "urn:ogc:def:crs:EPSG:26911")
    meta["crs_code"] = _text(root, ".//gmd:referenceSystemInfo//gmd:code/gco:CharacterString")

    # Resolution (metres)
    res = _text(root, ".//gmd:axisDimensionProperties//gmd:resolution/gco:Length")
    meta["resolution_m"] = float(res) if res else None

    # Grid dimensions
    cols = _text(root, ".//gmd:axisDimensionProperties[1]//gmd:dimensionSize/gco:Integer")
    rows_el = root.findall(".//gmd:axisDimensionProperties", NS)
    rows = None
    if len(rows_el) >= 2:
        rows = _text(rows_el[1], ".//gmd:dimensionSize/gco:Integer")
    meta["columns"] = int(cols) if cols else None
    meta["rows"] = int(rows) if rows else None

    # Bounding box
    bbox_el = root.find(".//gmd:EX_GeographicBoundingBox", NS)
    if bbox_el is not None:
        w = _text(bbox_el, "gmd:westBoundLongitude/gco:Decimal")
        e = _text(bbox_el, "gmd:eastBoundLongitude/gco:Decimal")
        s = _text(bbox_el, "gmd:southBoundLatitude/gco:Decimal")
        n = _text(bbox_el, "gmd:northBoundLatitude/gco:Decimal")
        try:
            meta["bbox_wgs84"] = [float(w), float(s), float(e), float(n)]
        except (TypeError, ValueError):
            meta["bbox_wgs84"] = None
    else:
        meta["bbox_wgs84"] = None

    # Keywords (place)
    place_kws = root.findall(
        ".//gmd:descriptiveKeywords/gmd:MD_Keywords"
        "[gmd:type/gmd:MD_KeywordTypeCode[@codeListValue='place']]"
        "/gmd:keyword/gco:CharacterString", NS
    )
    meta["place_keywords"] = [kw.text.strip() for kw in place_kws if kw.text]

    # Keywords (theme)
    theme_kws = root.findall(
        ".//gmd:descriptiveKeywords/gmd:MD_Keywords"
        "[gmd:type/gmd:MD_KeywordTypeCode[@codeListValue='theme']]"
        "/gmd:keyword/gco:CharacterString", NS
    )
    meta["theme_keywords"] = [kw.text.strip() for kw in theme_kws if kw.text]

    return meta


def metadata_from_raster(tif_path: Path) -> dict:
    """Fall-back: extract spatial metadata directly from a raster file."""
    with rasterio.open(tif_path) as src:
        native_bounds = src.bounds
        crs = src.crs
        # Transform bounds to EPSG:4326 for STAC
        if crs and not crs.is_geographic:
            w, s, e, n = transform_bounds(crs, "EPSG:4326",
                                          *native_bounds)
        else:
            w, s, e, n = native_bounds

        return {
            "crs_epsg": crs.to_epsg() if crs else None,
            "crs_wkt": crs.to_wkt() if crs else None,
            "bbox_wgs84": [w, s, e, n],
            "native_bounds": list(native_bounds),
            "resolution": src.res,
            "width": src.width,
            "height": src.height,
            "band_count": src.count,
            "dtype": str(src.dtypes[0]),
        }


# ── Parse metadata for every dataset ─────────────────────────────────────────
parsed_metadata: dict[str, dict] = {}   # dataset_name → combined metadata dict

for ds_name, entry in tqdm(dataset_catalog.items(), desc="Parsing metadata"):
    meta = {"dataset": ds_name, "folder": str(entry["folder"])}

    # Try XML first
    xml_files = entry["files"]["metadata"]
    xml_parsed = False
    for xf in xml_files:
        if xf.suffix.lower() == ".xml":
            try:
                xml_meta = parse_iso_xml(xf)
                meta.update(xml_meta)
                xml_parsed = True
                break
            except Exception as exc:
                meta["xml_parse_error"] = str(exc)

    # Supplement / fall back with raster metadata
    image_files = entry["files"]["image"]
    if image_files:
        try:
            raster_meta = metadata_from_raster(image_files[0])
            meta["raster"] = raster_meta
            # Fill gaps from raster if XML didn't provide them
            if not meta.get("bbox_wgs84"):
                meta["bbox_wgs84"] = raster_meta["bbox_wgs84"]
        except Exception as exc:
            meta["raster_read_error"] = str(exc)

    parsed_metadata[ds_name] = meta

# ── Summary ──────────────────────────────────────────────────────────────────
have_bbox = sum(1 for m in parsed_metadata.values() if m.get("bbox_wgs84"))
have_xml  = sum(1 for m in parsed_metadata.values() if m.get("title"))

print(f"\n{'═' * 60}")
print(f" Metadata Parsing Summary")
print(f"{'═' * 60}")
print(f"  Datasets parsed     : {len(parsed_metadata)}")
print(f"  With XML title      : {have_xml}")
print(f"  With bounding box   : {have_bbox}")
print(f"{'═' * 60}")

# Show one sample
sample_name = next(iter(parsed_metadata))
sample = parsed_metadata[sample_name]
print(f"\n  Sample — {sample_name}:")
for k in ["title", "date_published", "crs_code", "resolution_m", "bbox_wgs84"]:
    print(f"    {k:<20}: {sample.get(k)}")

## Step 7 — Validate Images: COG Check

A **Cloud-Optimized GeoTIFF (COG)** has:
- Internal tiling (typically 256×256 or 512×512 blocks)
- Overview pyramids (reduced-resolution layers)
- Proper byte ordering so HTTP range requests work efficiently

`rio_cogeo.cog_validate()` returns `(is_valid, errors, warnings)`. We classify each image as `valid_cog` or `needs_conversion`.

In [ ]:
# ── COG Validation ────────────────────────────────────────────────────────────

cog_status: dict[str, list] = {"valid_cog": [], "needs_conversion": []}

all_image_paths = [
    img
    for entry in dataset_catalog.values()
    for img in entry["files"]["image"]
]

print(f"Validating {len(all_image_paths)} image file(s) …\n")

for img_path in tqdm(all_image_paths, desc="COG validation"):
    try:
        is_valid, errors, warnings = cog_validate(str(img_path))
        if is_valid:
            cog_status["valid_cog"].append(img_path)
        else:
            cog_status["needs_conversion"].append(img_path)
    except Exception as exc:
        print(f"  ⚠ Could not validate {img_path.name}: {exc}")
        cog_status["needs_conversion"].append(img_path)

print(f"\n{'═' * 60}")
print(f" COG Validation Results")
print(f"{'═' * 60}")
print(f"  Already valid COGs  : {len(cog_status['valid_cog'])}")
print(f"  Need conversion     : {len(cog_status['needs_conversion'])}")
print(f"{'═' * 60}")

if cog_status["needs_conversion"]:
    print(f"\n  First few needing conversion:")
    for p in cog_status["needs_conversion"][:5]:
        print(f"    • {p.relative_to(DOWNLOAD_DIR)}")

## Step 8 — Convert Non-COG Images to COGs

For each image that failed the COG check, `rio_cogeo.cog_translate` creates a properly tiled, compressed COG with overview pyramids.

The conversion writes to a **temporary file** first, validates it, and only then proceeds to the replacement step. This avoids corrupting the working copy if conversion fails.

> **Note:** NAIP TIFs are typically ~400–500 MB uncompressed. LZW compression + tiling usually reduces size significantly. This step may take a while for large collections.

In [ ]:
# ── Convert non-COG images to COGs ────────────────────────────────────────────

output_profile = cog_profiles.get(COG_PROFILE_NAME)
output_profile.update({"blockxsize": COG_BLOCKSIZE, "blockysize": COG_BLOCKSIZE})

cog_conversion_results: list[dict] = []   # [{src, tmp, valid, error}]

to_convert = cog_status["needs_conversion"]
print(f"Converting {len(to_convert)} image(s) to COG …\n")

for img_path in tqdm(to_convert, desc="Creating COGs"):
    tmp_cog = img_path.parent / f"{img_path.stem}_cog_tmp{img_path.suffix}"
    result = {"src": img_path, "tmp": tmp_cog, "valid": False, "error": None}

    try:
        cog_translate(
            str(img_path),
            str(tmp_cog),
            output_profile,
            overview_level=OVERVIEW_LEVELS,
            overview_resampling=OVERVIEW_RESAMPLING,
            quiet=True,
        )

        # Validate the newly created COG
        is_valid, errors, warnings = cog_validate(str(tmp_cog))
        result["valid"] = is_valid
        if not is_valid:
            result["error"] = f"Validation failed: {errors}"

    except Exception as exc:
        result["error"] = str(exc)

    cog_conversion_results.append(result)

# ── Summary ──────────────────────────────────────────────────────────────────
ok_conversions = [r for r in cog_conversion_results if r["valid"]]
bad_conversions = [r for r in cog_conversion_results if not r["valid"]]

print(f"\n{'═' * 60}")
print(f" COG Conversion Results")
print(f"{'═' * 60}")
print(f"  Successfully converted : {len(ok_conversions)}")
print(f"  Failed                 : {len(bad_conversions)}")
print(f"{'═' * 60}")

if bad_conversions:
    print("\n⚠ Failures:")
    for r in bad_conversions:
        print(f"  {r['src'].name}: {r['error']}")

## Steps 9 & 10 — Replace Originals with COGs / Stage Originals for Disposal

These two operations happen together for each image:

1. **Move** the original (non-COG) image to `originals_for_disposal/`, preserving subfolder structure and generating a SHA-256 checksum.
2. **Rename** the temp COG to the original filename so all relative references (XML, sidecar files) remain valid.
3. **Re-validate** the in-place COG.

A disposal manifest CSV is written at the end, listing every staged file with its checksum, source path, and size.

In [ ]:
# ── Replace originals with COGs & stage originals for disposal ────────────────

disposal_manifest: list[dict] = []   # rows for the CSV manifest
replacement_errors: list[str] = []

for r in tqdm(cog_conversion_results, desc="Replacing / staging"):
    if not r["valid"]:
        # Clean up failed temp file if it exists
        if r["tmp"].exists():
            r["tmp"].unlink()
        continue

    src: Path = r["src"]     # original image in extracted folder
    tmp: Path = r["tmp"]     # validated temp COG

    # ── 1. Move original to disposal folder ──────────────────────────────
    # Preserve relative sub-path under DOWNLOAD_DIR
    rel = src.relative_to(DOWNLOAD_DIR)
    disposal_dest = ORIGINALS_DIR / rel
    disposal_dest.parent.mkdir(parents=True, exist_ok=True)

    try:
        # Checksum the original before moving
        orig_sha256 = sha256_file(src)
        orig_size = src.stat().st_size

        shutil.move(str(src), str(disposal_dest))

        disposal_manifest.append({
            "original_path": str(rel),
            "disposal_path": str(disposal_dest),
            "sha256": orig_sha256,
            "size_bytes": orig_size,
        })

    except Exception as exc:
        replacement_errors.append(f"Move failed for {src.name}: {exc}")
        continue

    # ── 2. Rename temp COG to original filename ──────────────────────────
    try:
        tmp.rename(src)   # same parent dir, same filename → seamless replacement
    except Exception as exc:
        replacement_errors.append(f"Rename failed for {tmp.name}: {exc}")
        continue

    # ── 3. Re-validate the in-place COG ──────────────────────────────────
    try:
        is_valid, _, _ = cog_validate(str(src))
        if not is_valid:
            replacement_errors.append(f"Post-replace validation failed: {src.name}")
    except Exception as exc:
        replacement_errors.append(f"Post-replace validate error: {src.name}: {exc}")

# ── Write disposal manifest CSV ──────────────────────────────────────────────
manifest_path = ORIGINALS_DIR / "disposal_manifest.csv"
with open(manifest_path, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["original_path", "disposal_path", "sha256", "size_bytes"])
    writer.writeheader()
    writer.writerows(disposal_manifest)

# ── Summary ──────────────────────────────────────────────────────────────────
print(f"\n{'═' * 60}")
print(f" Replacement & Disposal Summary")
print(f"{'═' * 60}")
print(f"  Originals staged for disposal : {len(disposal_manifest)}")
print(f"  Manifest written to           : {manifest_path}")
print(f"  Replacement errors            : {len(replacement_errors)}")
print(f"{'═' * 60}")

if replacement_errors:
    print("\n⚠ Errors:")
    for e in replacement_errors:
        print(f"    {e}")

## Step 11 — Build STAC Items for Each Dataset

For every extracted scene, a [STAC Item](https://github.com/radiantearth/stac-spec/blob/master/item-spec/item-spec.md) is created with:

- **ID** derived from the folder name (NAIP quad-ID + date)
- **Geometry & bbox** in EPSG:4326 (from XML or rasterio)
- **datetime** from the acquisition date encoded in the filename (YYYYMMDD)
- **Assets** pointing to the COG image and the ISO XML metadata file
- **Properties** including CRS, resolution, band count, and keywords from ISO XML
- **Extensions**: `proj` (projection) populated with EPSG code and native transform

Each Item JSON is written into its respective extracted folder.

In [ ]:
# ── Build STAC Items ──────────────────────────────────────────────────────────

import re

def _parse_naip_date(name: str) -> datetime | None:
    """Extract acquisition date from a NAIP filename like m_3611842_sw_11_060_20220704."""
    match = re.search(r"(\d{8})$", name)
    if match:
        return datetime.strptime(match.group(1), "%Y%m%d").replace(tzinfo=timezone.utc)
    return None


stac_items: dict[str, Item] = {}   # dataset_name → pystac.Item

for ds_name, meta in tqdm(parsed_metadata.items(), desc="Building STAC Items"):
    entry = dataset_catalog.get(ds_name)
    if entry is None:
        continue

    folder: Path = entry["folder"]

    # ── Geometry & bbox ──────────────────────────────────────────────────
    bbox = meta.get("bbox_wgs84")
    if not bbox or len(bbox) != 4:
        print(f"  ⚠ Skipping {ds_name}: no bounding box")
        continue

    geom = mapping(box(*bbox))  # GeoJSON Polygon from bbox

    # ── Datetime ─────────────────────────────────────────────────────────
    dt = _parse_naip_date(ds_name)
    if dt is None and meta.get("date_published"):
        try:
            dt = datetime.fromisoformat(meta["date_published"]).replace(tzinfo=timezone.utc)
        except Exception:
            dt = datetime.now(timezone.utc)

    # ── Properties ───────────────────────────────────────────────────────
    properties: dict = {}
    if meta.get("title"):
        properties["title"] = meta["title"]
    if meta.get("abstract"):
        properties["description"] = meta["abstract"]
    if meta.get("resolution_m"):
        properties["gsd"] = meta["resolution_m"]
    if meta.get("place_keywords"):
        properties["naip:place_keywords"] = meta["place_keywords"]
    if meta.get("theme_keywords"):
        properties["naip:theme_keywords"] = meta["theme_keywords"]

    # Projection extension properties
    raster = meta.get("raster", {})
    if raster.get("crs_epsg"):
        properties["proj:epsg"] = raster["crs_epsg"]
    elif meta.get("crs_code"):
        # Try to extract EPSG int from string like "urn:ogc:def:crs:EPSG:26911"
        epsg_match = re.search(r"(\d{4,6})$", meta["crs_code"])
        if epsg_match:
            properties["proj:epsg"] = int(epsg_match.group(1))

    if raster.get("width"):
        properties["proj:shape"] = [raster["height"], raster["width"]]
    if raster.get("native_bounds"):
        properties["proj:bbox"] = raster["native_bounds"]
    if raster.get("band_count"):
        properties["eo:bands_count"] = raster["band_count"]

    # ── Create STAC Item ─────────────────────────────────────────────────
    item = Item(
        id=ds_name,
        geometry=geom,
        bbox=bbox,
        datetime=dt,
        properties=properties,
    )

    # ── Assets ───────────────────────────────────────────────────────────
    # COG image(s)
    for img in entry["files"]["image"]:
        item.add_asset(
            key="image",
            asset=Asset(
                href=str(img.relative_to(folder)),
                media_type=MediaType.COG,
                roles=["data"],
                title="COG image",
            ),
        )

    # XML metadata
    for xf in entry["files"]["metadata"]:
        if xf.suffix.lower() == ".xml":
            item.add_asset(
                key="metadata",
                asset=Asset(
                    href=str(xf.relative_to(folder)),
                    media_type=MediaType.XML,
                    roles=["metadata"],
                    title="ISO 19115 metadata",
                ),
            )

    # ── Write Item JSON into its folder ──────────────────────────────────
    item_path = folder / f"{ds_name}.json"
    item.set_self_href(str(item_path))
    item.save_object()

    stac_items[ds_name] = item

print(f"\n{'═' * 60}")
print(f" STAC Item Summary")
print(f"{'═' * 60}")
print(f"  Items created: {len(stac_items)}")
print(f"  Written to   : <dataset_folder>/<dataset_name>.json")
print(f"{'═' * 60}")

## Step 12 — Create & Validate Top-Level STAC Catalog

The STAC Catalog sits at the **top level** of the downloads directory, alongside the ZIP files and extracted folders:

```
downloads/naip/
├── catalog.json                     ← STAC Catalog
├── m_3611842_sw_11_060_20220704/    ← extracted scene folder
│   ├── m_3611842_sw_11_060_20220704.tif   (COG)
│   ├── m_3611842_sw_11_060_20220704.xml
│   └── m_3611842_sw_11_060_20220704.json  (STAC Item)
├── m_3611842_sw_11_060_20220704.ZIP ← original ZIP
└── …
```

The catalog uses **self-contained** relative links so the entire directory can be moved or uploaded as a unit.

In [ ]:
# ── Create & validate STAC Catalog ────────────────────────────────────────────

catalog = Catalog(
    id=STAC_CATALOG_ID,
    title=STAC_CATALOG_TITLE,
    description=STAC_CATALOG_DESC,
)

# Add every Item as a child
for item in stac_items.values():
    catalog.add_item(item)

# Normalize all hrefs to relative paths rooted at DOWNLOAD_DIR
catalog.normalize_hrefs(str(DOWNLOAD_DIR.resolve()))

# Validate the catalog against the STAC spec
validation_errors = []
try:
    catalog.validate_all()
    print("✔ STAC catalog passed validation.")
except Exception as exc:
    validation_errors.append(str(exc))
    print(f"⚠ Validation warning: {exc}")

# Save as self-contained (all links are relative)
catalog.save(catalog_type=pystac.CatalogType.SELF_CONTAINED)

catalog_json_path = DOWNLOAD_DIR / "catalog.json"
print(f"\n{'═' * 60}")
print(f" STAC Catalog")
print(f"{'═' * 60}")
print(f"  ID          : {catalog.id}")
print(f"  Title       : {catalog.title}")
print(f"  Items       : {len(list(catalog.get_items(recursive=True)))}")
print(f"  Catalog file: {catalog_json_path}")
print(f"  File exists : {catalog_json_path.exists()}")
print(f"{'═' * 60}")

# Pretty-print the catalog JSON for inspection
if catalog_json_path.exists():
    with open(catalog_json_path) as f:
        print(json.dumps(json.load(f), indent=2)[:2000])
        print("… (truncated)" if catalog_json_path.stat().st_size > 2000 else "")

## Step 13 — Final Verification Report

End-to-end audit:

1. **ZIP integrity** — re-checksum every ZIP and compare to the pre-processing values to confirm they were never modified.
2. **Disposal folder** — verify all expected original files are present with correct checksums.
3. **COG status** — spot-check that extracted images are now valid COGs.
4. **STAC catalog** — confirm catalog.json exists and validates.
5. **Pipeline log** — write a summary log to the download folder.

In [ ]:
# ── Final Verification Report ─────────────────────────────────────────────────

report_lines: list[str] = []

def rpt(msg: str):
    """Print and collect report lines."""
    print(msg)
    report_lines.append(msg)


rpt(f"{'═' * 70}")
rpt(f" SDR-PREPARATION PIPELINE — VERIFICATION REPORT")
rpt(f" Generated: {datetime.now(timezone.utc).isoformat()}")
rpt(f"{'═' * 70}")

# ── 1. ZIP integrity ─────────────────────────────────────────────────────────
rpt(f"\n1. ZIP INTEGRITY CHECK")
zip_ok = 0
zip_changed = 0
for z in tqdm(zip_inventory, desc="Verifying ZIPs"):
    current_sha = sha256_file(z["path"])
    if current_sha == z["sha256"]:
        zip_ok += 1
    else:
        zip_changed += 1
        rpt(f"   ⚠ CHANGED: {z['name']}")
        rpt(f"     Before: {z['sha256']}")
        rpt(f"     After : {current_sha}")

rpt(f"   Verified unchanged : {zip_ok} / {len(zip_inventory)}")
rpt(f"   Changed/corrupted  : {zip_changed}")

# ── 2. Disposal folder ──────────────────────────────────────────────────────
rpt(f"\n2. DISPOSAL FOLDER")
rpt(f"   Location: {ORIGINALS_DIR.resolve()}")
disposal_files = list(ORIGINALS_DIR.rglob("*"))
disposal_files = [f for f in disposal_files if f.is_file() and f.name != "disposal_manifest.csv"]
rpt(f"   Original files staged : {len(disposal_files)}")
total_disposal_mb = sum(f.stat().st_size for f in disposal_files) / (1024 * 1024)
rpt(f"   Total size            : {total_disposal_mb:,.1f} MB")
rpt(f"   Manifest              : {ORIGINALS_DIR / 'disposal_manifest.csv'}")

# ── 3. COG spot-check ───────────────────────────────────────────────────────
rpt(f"\n3. COG SPOT-CHECK (re-validate a sample of extracted images)")
all_current_images = [
    img
    for entry in dataset_catalog.values()
    for img in entry["files"]["image"]
    if img.exists()
]
sample_size = min(5, len(all_current_images))
sample_imgs = all_current_images[:sample_size]
cog_ok = 0
for img in sample_imgs:
    try:
        is_valid, _, _ = cog_validate(str(img))
        status = "✔ valid COG" if is_valid else "✘ NOT a COG"
        if is_valid:
            cog_ok += 1
    except Exception as exc:
        status = f"✘ error: {exc}"
    rpt(f"   {img.name}: {status}")
rpt(f"   Sample pass rate: {cog_ok}/{sample_size}")

# ── 4. STAC catalog ─────────────────────────────────────────────────────────
rpt(f"\n4. STAC CATALOG")
rpt(f"   File exists : {catalog_json_path.exists()}")
rpt(f"   Items       : {len(stac_items)}")
if validation_errors:
    rpt(f"   ⚠ Validation issues: {validation_errors}")
else:
    rpt(f"   Validation  : ✔ passed")

# ── 5. Overall summary ──────────────────────────────────────────────────────
rpt(f"\n{'═' * 70}")
rpt(f" SUMMARY")
rpt(f"{'═' * 70}")
rpt(f"  ZIP files processed                : {len(zip_inventory)}")
rpt(f"  Datasets extracted                 : {len(extraction_results)}")
rpt(f"  Total files extracted              : {sum(len(v['files']) for v in extraction_results.values())}")
rpt(f"  Images already valid COGs          : {len(cog_status['valid_cog'])}")
rpt(f"  Images converted to COG            : {len(ok_conversions)}")
rpt(f"  Conversion failures                : {len(bad_conversions)}")
rpt(f"  Originals staged for disposal      : {len(disposal_manifest)}")
rpt(f"  STAC Items created                 : {len(stac_items)}")
rpt(f"  STAC catalog validation            : {'✔' if not validation_errors else '⚠'}")
rpt(f"  ZIP files unchanged                : {zip_ok} / {len(zip_inventory)}")
rpt(f"{'═' * 70}")

# ── Write log file ───────────────────────────────────────────────────────────
log_path = DOWNLOAD_DIR / "sdr_prep_report.log"
with open(log_path, "w") as f:
    f.write("\n".join(report_lines))
print(f"\nReport saved to {log_path}")